In [9]:
import pandas as pd
messy_orders = []
import json
import random
from datetime import datetime, timedelta

# Seed for reproducibility
random.seed(42)

statuses = ["COMPLETED", "PENDING", "CANCELLED", "completed", None]
cities = ["Seattle", "Chicago", "New York", "Austin", "SEATTLE", "chicago", None]

# Updated product pool with fruits, vegetables, and packaged goods
product_pool = [
    # Fruits
    {"id": "SKU_FRT_01", "name": "Organic Bananas (bunch)", "base_price": 2.49},
    {"id": "SKU_FRT_02", "name": "Hass Avocado", "base_price": 1.25},
    {"id": "SKU_FRT_03", "name": "Honeycrisp Apples (lb)", "base_price": 2.99},
    # Vegetables
    {"id": "SKU_VEG_01", "name": "Fresh Baby Spinach (10oz)", "base_price": 3.99},
    {"id": "SKU_VEG_02", "name": "Organic Carrots (2lb bag)", "base_price": 2.19},
    {"id": "SKU_VEG_03", "name": "Broccoli Crowns (lb)", "base_price": 1.99},
    # Packaged Goods
    {"id": "SKU_PKG_01", "name": "Almond Milk (Half Gallon)", "base_price": 3.89},
    {"id": "SKU_PKG_02", "name": "Greek Yogurt (32oz)", "base_price": 5.49},
    {"id": "SKU_PKG_03", "name": "Whole Wheat Bread", "base_price": 3.29},
    {"id": "SKU_PKG_04", "name": "Organic Penne Pasta (16oz)", "base_price": 2.79}
]

messy_grocery_orders = []

start_date = datetime(2026, 9, 1)

for i in range(1, 101):
    order_id = f"GROC-{1000 + i}"
    # Introduce duplicate order IDs occasionally to practice deduplication!
    if i in [15, 42, 88]:
        order_id = f"GROC-{1000 + (i - 5)}"
        
    customer_id = f"CUST-{random.randint(500, 550)}"
    order_date = (start_date + timedelta(days=random.randint(0, 20))).strftime("%Y-%m-%d %H:%M:%S")
    
    # Generate 1 to 4 items per grocery order
    num_items = random.randint(1, 4)
    items = []
    for _ in range(num_items):
        prod = random.choice(product_pool)
        # Messy pricing: sometimes string with $, sometimes float
        price_value = prod["base_price"] * random.uniform(0.9, 1.1)
        price = f"${price_value:.2f}" if random.random() > 0.5 else round(price_value, 2)
        
        items.append({
            "product_id": prod["id"],
            "product_name": prod["name"],
            "quantity": random.choice([1, 2, 3, "2", None]), # Messy types
            "item_price": price
        })

    order = {
        "order_id": order_id,
        "customer_id": customer_id,
        "order_timestamp": order_date,
        "status": random.choice(statuses),
        "shipping_address": {
            "city": random.choice(cities),
            "zip_code": random.choice(["98101", "60601", "10001", "invalid_zip", None])
        },
        "items": items
    }
    messy_grocery_orders.append(order)

# Wrap in an API-like response structure
api_payload = {
    "status": "success",
    "total_records": len(messy_grocery_orders),
    "data": messy_grocery_orders
}

# Save to a json file
with open("messy_grocery_orders.json", "w") as f:
    json.dump(api_payload, f, indent=4)

print("Successfully generated 'messy_grocery_orders.json' with 100 grocery records!")


Successfully generated 'messy_grocery_orders.json' with 100 grocery records!


In [10]:
api_payload

{'status': 'success',
 'total_records': 100,
 'data': [{'order_id': 'GROC-1001',
   'customer_id': 'CUST-540',
   'order_timestamp': '2026-09-04 00:00:00',
   'status': None,
   'shipping_address': {'city': 'Seattle', 'zip_code': None},
   'items': [{'product_id': 'SKU_VEG_02',
     'product_name': 'Organic Carrots (2lb bag)',
     'quantity': 1,
     'item_price': 2.08}]},
  {'order_id': 'GROC-1002',
   'customer_id': 'CUST-527',
   'order_timestamp': '2026-09-02 00:00:00',
   'status': None,
   'shipping_address': {'city': 'Chicago', 'zip_code': None},
   'items': [{'product_id': 'SKU_FRT_02',
     'product_name': 'Hass Avocado',
     'quantity': 1,
     'item_price': '$1.18'}]},
  {'order_id': 'GROC-1003',
   'customer_id': 'CUST-526',
   'order_timestamp': '2026-09-08 00:00:00',
   'status': None,
   'shipping_address': {'city': 'Seattle', 'zip_code': 'invalid_zip'},
   'items': [{'product_id': 'SKU_PKG_04',
     'product_name': 'Organic Penne Pasta (16oz)',
     'quantity': 2,
   

In [52]:

import pandas as pd 

df_silver_fresh_orders = pd.json_normalize(api_payload['data']) # flattens out dictioanries like ashipping address , we 
# need to apply trnsformations for items since its a list 
# 2. Explode the 'items' column FIRST so each item gets its own row
df_exploded = df_silver_fresh_orders.explode('items').reset_index(drop=True) 
df_exploded  
# 3. Now, pass the individual item dictionaries into json_normalize
df_items_only = pd.json_normalize(df_exploded['items'])
df_items_only  
df_final_silver_orders = pd.concat([df_exploded.drop(columns='items'), df_items_only], axis=1)
df_final_silver_orders # coverted all nested json into a dataframe 
cols_to_capitalize = ['status', 'shipping_address.city']

df_final_silver_orders[cols_to_capitalize] = df_final_silver_orders[cols_to_capitalize].apply(
    lambda x: x.str.capitalize()
)
df_final_silver_orders['shipping_address.zip_code'].fillna('Unknown') 
df_final_silver_orders 
# why is there invalid_zipcode for order_status = none but has item and quantity !,Shud those orders be considered pending or be dropped , all
#from same customer_id 
customer_order_count = df_final_silver_orders.groupby('customer_id')['order_id'].value_counts()
customer_order_count  
# filter invalid orders with orders_status or invalid_zipcode 

# Use .str on a single column, handling potential nulls with na=False
invalid_zips = df_final_silver_orders[
    df_final_silver_orders['shipping_address.zip_code'].str.contains('invalid', case=False, na=False)
]

invalid_zips[['order_id', 'shipping_address.zip_code']]

,order_id,shipping_address.zip_code
2,GROC-1003,invalid_zip
3,GROC-1003,invalid_zip
4,GROC-1003,invalid_zip
5,GROC-1003,invalid_zip
12,GROC-1006,invalid_zip
13,GROC-1006,invalid_zip
24,GROC-1011,invalid_zip
25,GROC-1011,invalid_zip
37,GROC-1018,invalid_zip
38,GROC-1018,invalid_zip


In [49]:
customer_order_count   


customer_id  order_id 
CUST-500     GROC-1018    2
             GROC-1021    1
CUST-502     GROC-1089    4
             GROC-1096    4
             GROC-1025    3
                         ..
CUST-548     GROC-1045    4
             GROC-1076    2
CUST-549     GROC-1006    2
             GROC-1100    2
CUST-550     GROC-1008    1
Name: count, Length: 100, dtype: int64

In [51]:
# filter invalid orders with orders_status or invalid_zipcode 

# Use .str on a single column, handling potential nulls with na=False
invalid_zips = df_final_silver_orders[
    df_final_silver_orders['shipping_address.zip_code'].str.contains('invalid', case=False, na=False)
]

invalid_zips[['order_id', 'shipping_address.zip_code']]

,order_id,shipping_address.zip_code
2,GROC-1003,invalid_zip
3,GROC-1003,invalid_zip
4,GROC-1003,invalid_zip
5,GROC-1003,invalid_zip
12,GROC-1006,invalid_zip
13,GROC-1006,invalid_zip
24,GROC-1011,invalid_zip
25,GROC-1011,invalid_zip
37,GROC-1018,invalid_zip
38,GROC-1018,invalid_zip
